# Tabular classification

Predict a category (yes/no, A/B/C) from rows of a spreadsheet or database.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/tabular-classification/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

AutoML (e.g. AutoGluon / Google Cloud AutoML / SageMaker Autopilot): Tries dozens of models for you. A good choice when you’re new or need a strong result quickly.

In [ ]:
%pip install -q autogluon

In [ ]:
# pip install autogluon
from autogluon.tabular import TabularPredictor
import pandas as pd

df = pd.read_csv("customers.csv")
predictor = TabularPredictor(label="churned").fit(df, time_limit=600)
print(predictor.leaderboard())

## Train your own model

Gradient-boosted trees (XGBoost / LightGBM): The best accuracy-to-effort ratio for tabular data. Fast, CPU-only, explainable with SHAP.

In [ ]:
%pip install -q lightgbm scikit-learn pandas

In [ ]:
# pip install lightgbm scikit-learn pandas
import pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("customers.csv")
X = df.drop(columns=["churned"])
y = df["churned"]
X = pd.get_dummies(X)  # simple categorical encoding

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, class_weight="balanced")
model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)])

pred = model.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, pred))
print(classification_report(y_te, pred > 0.5))
model.booster_.save_model("model.txt")